# 01 — Exploratory Bias Analysis

Visualise IFS and AIFS 2m temperature forecast bias: distributions, seasonal patterns, and geographic structure.

**Data access:** All queries run through `PipelineDB` (DuckDB) — no full tables loaded into memory.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))  # repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from nwp_census_eval.db import PipelineDB

# Override if data is synced locally:
# db = PipelineDB(aggregated_dir='/local/path/to/aggregated')
db = PipelineDB()
print('Registered views:', db.registered_views())

## Summary statistics by lead time

In [ ]:
ifs_stats = db.summary_stats(model='ifs')
aifs_stats = db.summary_stats(model='aifs')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, df) in zip(axes, [('IFS', ifs_stats), ('AIFS', aifs_stats)]):
    grp = df.groupby('lead_time')[['mean_bias', 'rmse']].mean()
    grp.plot(ax=ax, marker='o', markersize=4)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(f'{label} — mean bias & RMSE by lead time')
    ax.set_xlabel('Lead time (h)')
    ax.set_ylabel('K')
plt.tight_layout()

## Seasonal bias pattern (monthly × lead time)

In [ ]:
df_seasonal = db.query("""
    SELECT
        MONTH(valid_time) AS month,
        lead_time,
        AVG(bias)          AS mean_bias
    FROM ifs_bias
    GROUP BY MONTH(valid_time), lead_time
    ORDER BY month, lead_time
""")

pivot = df_seasonal.pivot(index='lead_time', columns='month', values='mean_bias')
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1, origin='lower')
ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{lt}h' for lt in pivot.index])
ax.set_title('IFS mean bias — month × lead time')
plt.colorbar(im, ax=ax, label='Bias (K)')
plt.tight_layout()

## Geographic distribution of mean bias

In [ ]:
# County-level mean bias for 24h lead
df_county = db.query("""
    SELECT geo_id, AVG(bias) AS mean_bias
    FROM ifs_bias
    WHERE lead_time = 24
    GROUP BY geo_id
""")
print(f'{len(df_county):,} counties | bias range: {df_county.mean_bias.min():.2f} to {df_county.mean_bias.max():.2f} K')
df_county['mean_bias'].hist(bins=60, figsize=(8, 4))
plt.axvline(0, color='r', lw=1, ls='--')
plt.xlabel('Mean bias (K)')
plt.title('IFS 24h lead — county mean bias distribution')